# Test de NMF pour l'analyse de topic

## Chargement des librairies nécessaires

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF


In [3]:
df=pd.read_csv("corpus_zola_nettoye.csv") # Chargement du DataFrame nettoyé à partir du fichier CSV

In [4]:
def segmenter_texte(text, taille=500): # On segmente le texte en segments de 300 mots
    mots = text.split() # On divise le texte en mots
    segments = [] # Liste pour stocker les segments de texte
    for i in range(0, len(mots), taille): # On boucle sur les mots par pas de "taille" (300 mots)
        segment = " ".join(mots[i:i+taille]) # On crée un segment en joignant les mots du segment
        if len(segment.split()) >= 100:  # On vérifie que le segment contient au moins 100 mots pour éviter les segments trop courts
            segments.append(segment) # On ajoute le segment à la liste des segments
    return segments # On retourne la liste des segments de texte

lignes = [] # Liste pour stocker les segments de texte avec leurs titres et IDs

for _, row in df.iterrows():
    titre = row["nom_fichier"] # On utilise le nom du fichier comme titre
    texte = row["texte_nettoye"] # On segmente le texte en segments de 300 mots
    segments = segmenter_texte(texte) # On ajoute chaque segment à la liste avec son titre et son ID
    
    for j, seg in enumerate(segments):
        
        
        lignes.append({ 
            "titre": titre, # Titre du texte (nom du fichier)
            "segment_id": j, # ID du segment (index du segment dans le texte) 
            "texte_segment": seg # Contenu du segment de texte
        })

df_segments = pd.DataFrame(lignes) # Création d'un DataFrame à partir de la liste de segments

df_segments

,titre,segment_id,texte_segment
0,Carnets d'enquetes - Emile Zola_clean.txt,0,bruit étourdissement impossibilité place fixit...
1,Carnets d'enquetes - Emile Zola_clean.txt,1,percevoir prendre peur fourre tapis table crie...
2,Carnets d'enquetes - Emile Zola_clean.txt,2,carreau couloir carré escalier couloir cruche ...
3,Carnets d'enquetes - Emile Zola_clean.txt,3,accessoire accrocher agrément amende appuyer e...
4,Carnets d'enquetes - Emile Zola_clean.txt,4,poteau juge numéro cheval drapeau poteau arriv...
...,...,...,...
3458,1894_1_Lourdes._clean.txt,129,refaire pensée beau voyage souvenir devoir con...
3459,1894_1_Lourdes._clean.txt,130,étendre bras croix unir membre membre bouche b...
3460,1894_1_Lourdes._clean.txt,131,sœur sœur charité corps rester exposé jour fou...
3461,1894_1_Lourdes._clean.txt,132,vie valoir suite audace opérer humanité fermer...


In [5]:
df_segments["nb_tokens_nettoyes"] = df_segments["texte_segment"].apply(lambda x: len(x.split())) # Calcul du nombre de tokens nettoyés pour chaque segment
df_segments["nb_tokens_nettoyes"].describe() # Affichage des statistiques descriptives du nombre de tokens nettoyés par segment (moyenne, écart-type, min, max, etc.)

count    3463.000000
mean      498.366157
std        20.880196
min       115.000000
25%       500.000000
50%       500.000000
75%       500.000000
max       500.000000
Name: nb_tokens_nettoyes, dtype: float64

In [6]:
df_segments

,titre,segment_id,texte_segment,nb_tokens_nettoyes
0,Carnets d'enquetes - Emile Zola_clean.txt,0,bruit étourdissement impossibilité place fixit...,500
1,Carnets d'enquetes - Emile Zola_clean.txt,1,percevoir prendre peur fourre tapis table crie...,500
2,Carnets d'enquetes - Emile Zola_clean.txt,2,carreau couloir carré escalier couloir cruche ...,500
3,Carnets d'enquetes - Emile Zola_clean.txt,3,accessoire accrocher agrément amende appuyer e...,500
4,Carnets d'enquetes - Emile Zola_clean.txt,4,poteau juge numéro cheval drapeau poteau arriv...,500
...,...,...,...,...
3458,1894_1_Lourdes._clean.txt,129,refaire pensée beau voyage souvenir devoir con...,500
3459,1894_1_Lourdes._clean.txt,130,étendre bras croix unir membre membre bouche b...,500
3460,1894_1_Lourdes._clean.txt,131,sœur sœur charité corps rester exposé jour fou...,500
3461,1894_1_Lourdes._clean.txt,132,vie valoir suite audace opérer humanité fermer...,500


In [7]:
vectorizer = TfidfVectorizer( 
    min_df=5,
    max_df=0.8
    )
# min_df=5 : Ignorer les termes qui apparaissent dans moins de 3 segments, car ils sont considérés comme peu informatifs.
# max_df=0.8 : Ignorer les termes qui apparaissent dans plus de 80% des segments, car ils sont trop fréquents et ne contribuent pas à différencier les segments.

X = vectorizer.fit_transform(df_segments["texte_segment"])  
#Appliquer le TF-IDF vectorizer sur les segments de texte nettoyés pour obtenir une matrice de caractéristiques (termes pondérés par leur importance dans les segments).

X.shape

(3463, 11810)

In [8]:
nmf = NMF(n_components=10, 
          random_state=42,
          init="nndsvda",
          solver="cd",
          max_iter=500,
          
          )
W = nmf.fit_transform(X)
H = nmf.components_


feature_names = vectorizer.get_feature_names_out()

def afficher_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topic {topic_idx+1} : {' | '.join(top_words)}")

afficher_topics(nmf, feature_names, 10)

Topic 1 : aimer | femme | jeune | amour | cœur | sentir | pensée | chambre | vie | amant
Topic 2 : peuple | école | travail | œuvre | vérité | instituteur | père | justice | frère | enfant
Topic 3 : soleil | arbre | blanc | eau | rue | ciel | herbe | fleur | rose | haut
Topic 4 : monsieur | salon | dame | femme | comte | porte | air | demander | jeune | sourire
Topic 5 : abbé | prêtre | curé | église | trouche | soutane | autel | jardin | évêque | sous
Topic 6 : prussien | soldat | route | armée | général | empereur | capitaine | corps | cheval | troupe
Topic 7 : coupeau | manger | père | table | bon | vieux | coup | pain | boire | rire
Topic 8 : train | machine | gare | voie | chef | mécanicien | heure | berline | fosse | charbon
Topic 9 : franc | argent | rue | maison | vendre | payer | fortune | affaire | mois | bourse
Topic 10 : docteur | malade | enfant | mère | sœur | médecin | grotte | guérir | père | monsieur


/Users/morganr/analyse_topic_zola/.venv/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
